# Customer Analytics: Propensity, Churn, Retention & CLV

- [Four Questions, Four Models](#Four-Questions,-Four-Models)
  - [Propensity: who is likely to act](#Propensity:-who-is-likely-to-act)
  - [Churn: define the label first](#Churn:-define-the-label-first)
  - [Retention curves and cohorts](#Retention-curves-and-cohorts)
  - [CLV: what a customer is worth](#CLV:-what-a-customer-is-worth)
  - [Uplift: who to treat](#Uplift:-who-to-treat)
  - [From scores to decisions: budgets](#From-scores-to-decisions:-budgets)
  - [Backtesting a targeting policy offline](#Backtesting-a-targeting-policy-offline)
- [Blogs](#Blogs)
  - [Pitfalls](#Pitfalls)

### Four Questions, Four Models

Propensity, churn, retention and CLV all start from the same customer table, but each answers a different decision.

| Question | What is modelled | Decision it drives |
|---|---|---|
| Propensity | P(customer takes an action in the next N days) | whom to contact for a cross-sell |
| Churn | P(customer leaves in the next N days), or time until they leave | whom to try to retain |
| Retention | survival curve S(t) per cohort | how fast customers leave, and whether a change helped |
| CLV | expected discounted margin over the customer's lifetime | how much to spend to acquire or keep a customer |
| Uplift | the incremental effect of an action on one customer | whom to treat |

A propensity or churn score ranks customers by how likely they are to act or leave. It does not say where an action changes the outcome. That is uplift, covered near the end.

#### Propensity: who is likely to act

Define the label with a window ("buys within 30 days of the snapshot") and build features only from before the snapshot. Recency, frequency and monetary value (RFM) plus tenure are the standard starting features. A gradient-boosted model or logistic regression works. Judge it at the contact budget (lift and cumulative capture in the top deciles), not by AUC alone.

Synthetic customers with a known buying pattern:

In [1]:
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(0)
n = 20000
cust = pd.DataFrame({
    "recency_days": rng.integers(1, 365, n),
    "frequency_90d": rng.poisson(3, n),
    "monetary_90d": rng.gamma(2.0, 40.0, n),
    "tenure_months": rng.integers(1, 120, n),
})
logit = (-2.2 - 0.008 * cust.recency_days + 0.25 * cust.frequency_90d
         + 0.004 * cust.monetary_90d + 0.004 * cust.tenure_months)
cust["bought"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))

X, y = cust.drop(columns="bought"), cust["bought"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
model = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.08, random_state=0).fit(Xtr, ytr)
p = model.predict_proba(Xte)[:, 1]
print("AUC:", round(roc_auc_score(yte, p), 3), "| base rate:", round(yte.mean(), 3))

AUC: 0.748 | base rate: 0.116


In [2]:
df = pd.DataFrame({"p": p, "y": yte.values})
df["decile"] = pd.qcut(df.p.rank(method="first", ascending=False), 10, labels=range(1, 11))
tbl = df.groupby("decile", observed=True).agg(customers=("y", "size"), buyers=("y", "sum"))
tbl["rate"] = tbl.buyers / tbl.customers
tbl["lift"] = tbl.rate / df.y.mean()
tbl["cum_capture"] = tbl.buyers.cumsum() / tbl.buyers.sum()
print(tbl.round(3))

        customers  buyers   rate   lift  cum_capture
decile                                              
1             600     192  0.320  2.747        0.275
2             600     137  0.228  1.960        0.471
3             600     102  0.170  1.459        0.617
4             600      77  0.128  1.102        0.727
5             600      48  0.080  0.687        0.795
6             600      60  0.100  0.858        0.881
7             600      31  0.052  0.443        0.926
8             600      28  0.047  0.401        0.966
9             600      15  0.025  0.215        0.987
10            600       9  0.015  0.129        1.000


Read the table top-down. `lift` is the decile's buy rate divided by the overall rate, and `cum_capture` is the share of all buyers reached if you contact deciles 1 to k. With a budget for 20% of customers, `cum_capture` at decile 2 is what the campaign actually gets.

#### Churn: define the label first

Most churn projects fail on the label, not the model. In a contract business (subscription, loan) churn is an explicit cancellation. In a non-contract business (retail, app) it is inactivity, so you must choose a window, for example "no purchase in 90 days". Then build features only from before the snapshot, with a gap before the label window so that signals created by the leaving itself (a cancellation ticket) do not leak in:
```
|---- feature window (90d) ----|-- gap (7d) --|---- label window (30d) ----|
                          snapshot date
```
Customers who have not left yet are not "non-churners". They are censored: you only know they survived so far. Counting them as retained biases retention upward for recent customers. Survival analysis handles this. The Kaplan-Meier estimator below multiplies, at each churn time, the fraction who survived it among those still at risk.

Simulated customers with exponential lifetimes (mean 14 months) who joined at different dates, so each is observed for a different length of time:

In [3]:
rng = np.random.default_rng(1)
n = 2000
true_t = rng.exponential(scale=14, size=n)            # months until churn
obs_end = rng.uniform(3, 24, size=n)                  # months each customer has been observed
time = np.minimum(true_t, obs_end)
event = (true_t <= obs_end).astype(int)               # 1 = churned, 0 = still active (censored)
print("churned:", event.sum(), "| still active (censored):", (1 - event).sum())

def kaplan_meier(time, event):
    order = np.argsort(time); t = time[order]; e = event[order]
    at_risk, s, rows = len(t), 1.0, []
    for ti in np.unique(t):
        m = t == ti
        d = e[m].sum()
        if d: s *= 1 - d / at_risk
        rows.append((ti, s)); at_risk -= m.sum()
    return pd.DataFrame(rows, columns=["t", "S"])

km = kaplan_meier(time, event)
print("month | Kaplan-Meier | true exp(-t/14)")
for mth in (3, 6, 12, 18):
    print(f"{mth:5d} | {km[km.t <= mth].S.iloc[-1]:12.3f} | {np.exp(-mth / 14):.3f}")
print("naive 'share not churned':", round(1 - event.mean(), 3), "(mixes customers observed for 3 and 24 months)")

churned: 1149 | still active (censored): 851
month | Kaplan-Meier | true exp(-t/14)
    3 |        0.806 | 0.807
    6 |        0.654 | 0.651
   12 |        0.418 | 0.424
   18 |        0.268 | 0.276
naive 'share not churned': 0.426 (mixes customers observed for 3 and 24 months)


Kaplan-Meier tracks the true survival curve, while the naive share of customers who have not churned mixes people observed for 3 months with people observed for 24.

#### Retention curves and cohorts

A cohort table groups customers by signup month and shows the share still active at each age. Read down a column to compare cohorts at the same age. A change that helped shows up as higher curves in newer cohorts, but check that the customer mix did not change at the same time.

In [4]:
rng = np.random.default_rng(2)
rows = []
for cohort in range(6):                                # signup month 0..5
    for _ in range(500):
        life = int(rng.geometric(0.25))                # months the customer stays active
        rows.extend((cohort, m) for m in range(life))
act = pd.DataFrame(rows, columns=["cohort", "months_since_signup"])
sizes = act[act.months_since_signup == 0].groupby("cohort").size()
ret = act.groupby(["cohort", "months_since_signup"]).size().unstack(fill_value=0)
print(ret.div(sizes, axis=0).round(2).iloc[:, :6])

months_since_signup    0     1     2     3     4     5
cohort                                                
0                    1.0  0.71  0.54  0.40  0.29  0.22
1                    1.0  0.78  0.58  0.41  0.30  0.21
2                    1.0  0.76  0.56  0.44  0.36  0.27
3                    1.0  0.75  0.57  0.43  0.30  0.24
4                    1.0  0.75  0.54  0.41  0.30  0.22
5                    1.0  0.76  0.56  0.43  0.30  0.23


#### CLV: what a customer is worth

For a contract business with constant monthly retention `r`, monthly margin `m` and monthly discount rate `d`, the discounted sum has a closed form:

CLV = m * (1 + d) / (1 + d - r)

Worked example: margin $10 a month, retention 0.90, discount 0.01 a month. For a non-contract business there is no cancellation event, so CLV is estimated either with probabilistic models (BG/NBD for purchase counts plus Gamma-Gamma for order value) or by regressing the next 12 months of value on customer features.

In [5]:
m, r, d = 10.0, 0.90, 0.01
closed = m * (1 + d) / (1 + d - r)
summed = sum(m * (r / (1 + d)) ** t for t in range(2000))
print("closed form:", round(closed, 2), "| summed:", round(summed, 2))
for r in (0.80, 0.85, 0.90, 0.95):
    print(f"retention {r:.2f} -> CLV {m * (1 + d) / (1 + d - r):7.1f}")

closed form: 91.82 | summed: 91.82
retention 0.80 -> CLV    48.1
retention 0.85 -> CLV    63.1
retention 0.90 -> CLV    91.8
retention 0.95 -> CLV   168.3


CLV is very sensitive to retention: moving monthly retention from 0.90 to 0.95 almost doubles it. That is why a small retention gain can be worth more than a large acquisition discount, and why CLV is used to cap what you pay to acquire a customer.

#### Uplift: who to treat

Customers split into four groups by how they respond to an action such as an offer:

| Group | Buys without offer | Buys with offer | Treat? |
|---|---|---|---|
| Persuadables | no | yes | yes |
| Sure things | yes | yes | no, a wasted discount |
| Lost causes | no | no | no |
| Sleeping dogs | yes | no | no, the offer hurts |

A response model ranks by P(buy | offer), so it puts sure things first. An uplift model ranks by P(buy | offer) - P(buy | no offer), estimated from randomized data. The simplest version trains one model on treated customers and one on control (a two-model, or T-learner, approach) and subtracts.

Synthetic randomized offer where highly engaged customers buy anyway, price-sensitive customers respond, and a small group is put off:

In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier as HGB

rng = np.random.default_rng(3)
n = 40000
X = pd.DataFrame({"engagement": rng.normal(size=n), "price_sensitivity": rng.normal(size=n)})
treat = rng.integers(0, 2, n)                                       # randomized offer
base = 1 / (1 + np.exp(-(-1.5 + 1.0 * X.engagement)))               # engaged customers buy anyway
effect = 0.15 * (X.price_sensitivity > 0.5) - 0.08 * (X.price_sensitivity < -1.0)
y = rng.binomial(1, np.clip(base + treat * effect, 0, 1))

tr = np.arange(n) < 28000; te = ~tr
m1 = HGB(max_iter=100, random_state=0).fit(X[tr & (treat == 1)], y[tr & (treat == 1)])
m0 = HGB(max_iter=100, random_state=0).fit(X[tr & (treat == 0)], y[tr & (treat == 0)])
Xte, tte, yte = X[te], treat[te], y[te]
uplift = m1.predict_proba(Xte)[:, 1] - m0.predict_proba(Xte)[:, 1]
response = m1.predict_proba(Xte)[:, 1]
random_score = np.random.default_rng(4).random(len(uplift))

def incremental(score, frac=0.3):
    k = int(len(score) * frac); idx = np.argsort(-score)[:k]
    t, yy = tte[idx], yte[idx]
    return (yy[t == 1].mean() - yy[t == 0].mean()) * k              # incremental conversions among those selected

for name, s in [("random", random_score), ("response model", response), ("uplift model", uplift)]:
    print(f"{name:15s}: {incremental(s):7.0f} incremental conversions from treating the top 30%")

random         :      85 incremental conversions from treating the top 30%
response model :     257 incremental conversions from treating the top 30%
uplift model   :     412 incremental conversions from treating the top 30%


The response model beats random, because price-sensitive customers also buy more, but it spends part of the budget on engaged customers who would buy anyway. Ranking by uplift concentrates the offer on persuadables and gains clearly more. Evaluate uplift models with held-out randomized data (Qini or uplift-at-k curves), never with accuracy on the outcome.

#### From scores to decisions: budgets

A score does not decide who gets an action. A decision combines the expected incremental value, the cost of the action and the budget. Treat a customer only when expected incremental margin exceeds the cost, even if budget is left over, because spending the rest just destroys value. With many treatments and separate budgets this becomes a knapsack problem (see Uber below).

In [7]:
margin_per_conversion, cost = 40.0, 5.0
net = uplift * margin_per_conversion - cost                         # expected net value per customer if treated
budget = 1500.0
order = np.argsort(-net)
chosen = [i for i in order if net[i] > 0][: int(budget // cost)]
print(f"treat {len(chosen)} of {len(net)} customers | spend ${len(chosen) * cost:,.0f} of ${budget:,.0f} | expected net value ${net[chosen].sum():,.0f}")

treat 300 of 12000 customers | spend $1,500 of $1,500 | expected net value $1,561


#### Backtesting a targeting policy offline

Before a live nudge, you want to know what a prioritized-targeting policy would have achieved. If past treatment was randomized (even a small random holdout), you can estimate any policy's value from the log with inverse propensity scoring: keep only the customers whose logged treatment matches what the new policy would have done, and weight their outcomes by 1 divided by the probability of that treatment. Without randomization in the log, the estimate is biased by whoever was targeted before.

Report the lift over a baseline policy (treat nobody, or the current targeting) with an uncertainty range, and check that the policy's coverage overlaps the logged treatments, since weights explode where the log rarely did what the policy wants.

In [1]:
import numpy as np
rng = np.random.default_rng(5)
n = 80000
x1, x2 = rng.normal(size=n), rng.normal(size=n)
treat = rng.random(n) < 0.5                                                  # logged policy: random 50% got the nudge
base = 1 / (1 + np.exp(-(-2.2 + 0.8 * x1)))                                  # engaged customers adopt anyway
lift = 0.10 * (x2 > 0.5)                                                     # only part of the base responds to the nudge
p_adopt = lambda t: np.clip(base + t * lift, 0, 1)
y = rng.random(n) < p_adopt(treat)

policies = {"treat nobody": np.zeros(n, bool), "target by propensity (x1 high)": x1 > 0.5, "target by responsiveness (x2 high)": x2 > 0.5}
for name, pol in policies.items():
    agree = treat == pol
    ips = np.mean(agree * y / 0.5)                                           # inverse propensity score estimate of adoption rate
    truth = p_adopt(pol.astype(float)).mean()                                # known only because this is a simulation
    print(f"{name:36s} | IPS estimate {ips:.4f} | true value {truth:.4f}")

treat nobody                         | IPS estimate 0.1218 | true value 0.1221
target by propensity (x1 high)       | IPS estimate 0.1324 | true value 0.1316
target by responsiveness (x2 high)   | IPS estimate 0.1537 | true value 0.1530


### Blogs

- **Uber, causal inference for promotions.** Uber uses heterogeneous treatment effects and uplift modelling to find which groups benefit most from a treatment, and difference-in-differences and inverse probability of treatment weighting when assignment was not randomized. The point is to estimate what a campaign caused, not what correlates with it. [Using Causal Inference to Improve the Uber User Experience](https://www.uber.com/us/en/blog/causal-inference-at-uber/)
- **Uber, Tarot.** Uber's uplift models predict ROI and cost for each user and incentive pairing, but prediction is not enough because each business unit has an absolute quarterly budget. Uber pairs the uplift models with a budget pacer and a multiple-knapsack optimizer. Budget utilization rose from about 68% to 99.99%, and after switching from a linear-programming solver to Google OR-Tools CP-SAT, a 100,000-user problem solved in minutes. [Beyond Prediction: Solving the Multiple Knapsack Problem at Scale](https://www.uber.com/us/en/blog/solving-multiple-knapsack/)
- **Airbnb, lifetime value.** Airbnb defines baseline listing LTV as the bookings a listing will make in the next 365 days, and incremental LTV as baseline minus cannibalized bookings that would have happened elsewhere on the platform. When the pandemic broke the training data, they shortened training windows, added geographic granularity and moved to LightGBM for high-cardinality features. [How Airbnb Measures Listing Lifetime Value](https://airbnb.tech/ai-ml/how-airbnb-measures-listing-lifetime-value/)
- **Booking.com, 150 production models.** Booking.com's KDD 2019 paper draws six lessons from about 150 customer-facing models, including that model performance is not the same as business performance and that business impact should be tested with randomized controlled trials. [150 Successful Machine Learning Models: 6 Lessons Learned at Booking.com](https://www.semanticscholar.org/paper/150-Successful-Machine-Learning-Models:-6-Lessons-Bernardi-Mavridis/07d2ee90b475d2c0d1ed91f6485e86784f61189f)

#### Pitfalls

- A vague churn or propensity label. Fix the window and the definition with the business owner before modelling.
- Leakage from features built after the snapshot, such as cancellation tickets or post-label activity.
- Counting censored customers as retained.
- Targeting by propensity when the goal is incremental impact: you pay discounts to sure things and annoy sleeping dogs.
- Optimizing model metrics instead of the business metric, and not validating with a randomized holdout.
- Ignoring cannibalization: a retained or acquired customer may only be moved from another product or channel.